In this notebook, we will use both Elastic Search and MinSearch to evaluate our FAQ documents and search engine. 

In [5]:
import json 

with open("docs_with_ids.json", "rt") as f:
    documents = json.load(f) 

print(json.dumps(documents[0], indent=2))

{
  "text": "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  \u201cOffice Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon\u2019t forget to register in DataTalks.Club's Slack and join the channel.",
  "section": "General course-related questions",
  "question": "Course - When will the course start?",
  "course": "data-engineering-zoomcamp",
  "id": "c02e79ef"
}


In [6]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch("httsp://localhost:9200")

index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},  # course and id are keywords
            "id": {"type": "keyword"},
        }
    }
}

index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [7]:
from tqdm.auto import tqdm

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 948/948 [00:04<00:00, 209.85it/s]


In [30]:
def elastic_search(query, course):
    search_query = {
        "size": 5,  # limiting to 5 results
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": course
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []

    for hit in response["hits"]["hits"]:
        result_docs.append(hit["_source"])

    return result_docs

In [9]:
# sample search

elastic_search(
    query="I just discovered the course. Can I still join?",
    course="data-engineering-zoomcamp"
)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp',
  'id': '7842b56a'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp',
  'id': '63394d91'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it fin

In [10]:
import pandas as pd
from tqdm.auto import tqdm

In [11]:
df_ground_truth = pd.read_csv("ground-truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [12]:
ground_truth[0]

{'question': 'When does the course begin?',
 'course': 'data-engineering-zoomcamp',
 'document': 'c02e79ef'}

Now we will check for relevance of our ground truth data with main FAQ document to check whether an input query matches or not.

In [13]:
relevance_total = []

for query in tqdm(ground_truth):
    doc_id = query["document"]
    results = elastic_search(query=query["question"], course=query["course"]) # it returns top 5 relevant documents
    relevance = [d["id"] == doc_id for d in results]
    relevance_total.append(relevance)

100%|██████████| 4627/4627 [00:19<00:00, 239.90it/s]


In [14]:
relevance_total[:10]

[[True, False, False, False, False],
 [False, False, False, False, False],
 [False, False, False, False, False],
 [False, False, False, True, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False]]

> Will be using metrics called Hit Rate (Recall) and Mean Reciprocal rank for evaluating the search engine.

Hit Rate - We will check for atleast one True in each response and consider as hit (1) for that and get all values of hit and divide by total queries.

In [15]:
def hit_rate(relevance_total):
    hits = 0 
    for item in relevance_total:
        if True in item:
            hits += 1 
            
    return hits / len(relevance_total)


In [16]:
# calculate hit rate

hit_rate(relevance_total)

0.9152798789712556

In [17]:
def mean_reciprocal_rank(relevance_total):
    total_value = 0.0
    
    for item in relevance_total:
        for i, res in enumerate(item):
            if res == True:
                total_value = total_value + 1 / (i + 1)  # i + 1 because rank starts from 1
                
            
    return total_value / len(relevance_total)

In [18]:
# calculate MRR

mean_reciprocal_rank(relevance_total)

0.8366003890209635

The search has very good hit rate and good MRR as sometimes required document is not at the first hit (which is not in first place of top 5 results produced by elastic search).

Let's evaluate the same metrics for MinSearch engine as well and compare the results

In [20]:
import minsearch 

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"],
)

index.fit(documents)

In [21]:
def minsearch_search(query, course):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results

In [22]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

100%|██████████| 4627/4627 [00:14<00:00, 318.10it/s]


In [26]:
print(f"Min Search evaluation results: Hit rate {hit_rate(relevance_total):.2f}, MRR {mean_reciprocal_rank(relevance_total):.2f}")

Min Search evaluation results: Hit rate 0.77, MRR 0.66


> From the evalaution metrics, it is clear that Elastic Search engine works better than locally developed Min Search engine.

Making the above function more generic to evaluate


In [27]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mean_reciprocal_rank(relevance_total),
    }

In [31]:
evaluate(ground_truth, lambda q: elastic_search(q['question'], q['course']))

100%|██████████| 4627/4627 [00:10<00:00, 425.23it/s]


{'hit_rate': 0.7395720769397017, 'mrr': 0.6029788920106625}

In [29]:
evaluate(ground_truth, lambda q: minsearch_search(q['question'], q['course']))

100%|██████████| 4627/4627 [00:14<00:00, 308.53it/s]


{'hit_rate': 0.7722066133563864, 'mrr': 0.661454506159499}

Final Remarks:

* Ground truth dataset might required more cleaning.
* Generate using human data annotator instead of LLM.
* We may need more data.
